# Electrical Grid Stability — 能量泛函 QUBO
## 7×7 网格 FEM + Energy-Form QUBO + CIM 单次真机求解
PCA 降维 → 2D Poisson → $E(u)=\frac{1}{2}u^T K u - u^T F$ → CIM

In [1]:
import numpy as np
import pandas as pd
import kaiwu as kw
import warnings, json, os
warnings.filterwarnings('ignore')
kw.common.CheckpointManager.save_dir = '/tmp'
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BIT_WIDTH = 8
OUTPUT_DIR = 'D:/QPDE/pde+pinn/outputs_gs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('模块加载完成')

模块加载完成


In [2]:
# ===== 加载 Electrical Grid Stability 数据 =====
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00471/Data_for_UCI_named.csv'
df = pd.read_csv(url)
# stabf: 'stable'/'unstable' → ±1
X = df.drop(columns=['stab', 'stabf']).values.astype(float)
y_raw = df['stabf'].values
y = np.where(y_raw == 'unstable', 1, -1)  # unstable=+1, stable=-1
# 抽样 1000 点加速 PCA (10000 太多)
rng = np.random.default_rng(42)
idx_sample = rng.choice(len(y), min(2000, len(y)), replace=False)
X_sample, y_sample = X[idx_sample], y[idx_sample]
print(f'全量: {X.shape}, 抽样: {X_sample.shape}, 正类(unstable): {(y_sample==1).sum()}, 负类(stable): {(y_sample==-1).sum()}')

全量: (10000, 12), 抽样: (2000, 12), 正类(unstable): 1310, 负类(stable): 690


In [3]:
# ===== PCA 降维到 2D =====
scaler = StandardScaler().fit(X_sample)
X_scaled = scaler.transform(X_sample)
X_2d = PCA(n_components=2).fit_transform(X_scaled)
for j in range(2):
    lo, hi = X_2d[:,j].min(), X_2d[:,j].max()
    X_2d[:,j] = (X_2d[:,j] - lo) / (hi - lo) * 0.7 + 0.15

N = 7; h = 1.0 / (N - 1)
grid_x = np.linspace(0, 1, N); grid_y = np.linspace(0, 1, N)
GX, GY = np.meshgrid(grid_x, grid_y, indexing='ij')
nodes_2d = np.column_stack([GX.ravel(), GY.ravel()])
boundary = (GX.ravel()==0) | (GX.ravel()==1) | (GY.ravel()==0) | (GY.ravel()==1)
internal = ~boundary
print(f'网格: {N}x{N}={N*N} 节点, 内部={internal.sum()}')

网格: 7x7=49 节点, 内部=25


In [4]:
sigma = 0.12
f = np.zeros(N*N)
for idx in range(len(y_sample)):
    dist2 = np.sum((nodes_2d - X_2d[idx])**2, axis=1)
    f += y_sample[idx] * np.exp(-dist2 / (2*sigma**2))

idx_map = -np.ones(N*N, dtype=int); idx_map[internal] = np.arange(internal.sum())
n_i = internal.sum()
K_ii = np.zeros((n_i, n_i))
for i in range(1, N-1):
    for j in range(1, N-1):
        k = i * N + j; ki = idx_map[k]
        K_ii[ki, ki] = 4.0
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            nk = (i+di) * N + (j+dj)
            if internal[nk]: K_ii[ki, idx_map[nk]] = -1.0

rhs = h**2 * f[internal]
print(f'K_ii: {K_ii.shape}, κ(K_ii)={np.linalg.cond(K_ii):.2e}')

K_ii: (25, 25), κ(K_ii)=1.39e+01


In [5]:
u_ref = np.linalg.solve(K_ii, rhs)
margin = 0.5; rng = max(u_ref.max()-u_ref.min(), 0.1)
lb = u_ref.min() - margin * rng; ub = u_ref.max() + margin * rng
print(f'参考解范围: [{u_ref.min():.4f}, {u_ref.max():.4f}], 界: [{lb:.4f}, {ub:.4f}]')

参考解范围: [1.4869, 9.9059], 界: [-2.7226, 14.1154]


In [6]:
def build_energy_qubo(K, r, bw, lb, ub):
    n = K.shape[0]; nvar = n * bw
    scale = (ub - lb) / (2**bw - 1)
    s = scale * np.array([2**k for k in range(bw)])
    ssT = np.outer(s, s); c = lb * np.ones(n); w = K @ c - r
    Q = np.zeros((nvar, nvar))
    for i in range(n):
        for j in range(i, n):
            aij = K[i,j]
            if abs(aij) < 1e-15: continue
            ri, rj = i*bw, j*bw; blk = 0.5 * aij * ssT
            Q[ri:ri+bw, rj:rj+bw] += blk
            if i != j: Q[rj:rj+bw, ri:ri+bw] += blk.T
    for i in range(n): Q[i*bw:(i+1)*bw, i*bw:(i+1)*bw] += np.diag(w[i] * s)
    return Q.astype(np.float32), nvar, scale

Q_float, nvar, scale = build_energy_qubo(K_ii, rhs, BIT_WIDTH, lb, ub)
Q_qubo = kw.qubo.adjust_qubo_matrix_precision(Q_float)
print(f'QUBO: {nvar}x{nvar}, 值: [{Q_qubo.min():.1f}, {Q_qubo.max():.1f}]')

QUBO: 200x200, 值: [-384.0, 760.0]


In [7]:
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
vars_ = [f'x[{i}]' for i in range(ising_mat.shape[0])]
ising_model = kw.ising.IsingModel(variables=vars_, ising_matrix=ising_mat, bias=ising_bias)
opt = kw.cim.CIMOptimizer(task_name='gs_energy', task_mode='quota')
opt.solve(ising_model.get_matrix())
print(f'gs_energy 已提交, Ising: {ising_mat.shape}')

[2026-05-20 18:12:06] [INFO    ] [kaiwu.cim._optimizer_adapter:5] - Task submit successfully, waiting for data validation. Task name: gs_energy
gs_energy 已提交, Ising: (201, 201)


In [8]:
sol = opt.solve(ising_model.get_matrix())
print(f'返回: {sol.shape}')
sols_bin = (sol[:,:-1] * sol[:,-1:]+1)/2
energies = np.array([z@Q_qubo@z for z in sols_bin])
z_best = sols_bin[np.argmin(energies)]
u_quantum_i = np.zeros(n_i)
s = scale * np.array([2**k for k in range(BIT_WIDTH)])
for i in range(n_i): u_quantum_i[i] = np.dot(s, z_best[i*BIT_WIDTH:(i+1)*BIT_WIDTH]) + lb
print(f'量子解: [{u_quantum_i.min():.4f}, {u_quantum_i.max():.4f}], 能量: {energies.min():.1f}')

[2026-05-20 18:13:22] [INFO    ] [kaiwu.cim._optimizer_adapter:1] - Task completed: gs_energy
返回: (10, 201)
量子解: [0.7770, 9.6253], 能量: -2664.0


In [9]:
u_quantum = np.zeros(N*N); u_quantum[internal] = u_quantum_i
np.save(f'{OUTPUT_DIR}/preset_nodes_energy.npy', nodes_2d)
np.save(f'{OUTPUT_DIR}/preset_values_energy.npy', u_quantum)
np.save(f'{OUTPUT_DIR}/X_2d.npy', X_2d); np.save(f'{OUTPUT_DIR}/y.npy', y_sample)
np.save(f'{OUTPUT_DIR}/K_ii.npy', K_ii); np.save(f'{OUTPUT_DIR}/rhs.npy', rhs)
np.save(f'{OUTPUT_DIR}/internal_mask.npy', internal)
rmse = np.sqrt(np.mean((u_quantum_i - u_ref)**2))
meta = {'method':'energy','n_internal':int(n_i),'qubo_size':int(nvar),'rmse_vs_classical':float(rmse),'best_energy':float(energies.min())}
with open(f'{OUTPUT_DIR}/meta_energy.json','w') as f: json.dump(meta, f, indent=2)
print(f'预设已保存, RMSE vs 经典: {rmse:.6e}')

预设已保存, RMSE vs 经典: 4.757895e-01
